# Kaggle GPU Runner — 1 Mô Hình / Phiên (Không bao giờ dính giới hạn 12h)
Đổi biến **`EXP`** ở Cell 1 sang đúng thí nghiệm muốn chạy (mỗi phiên chỉ mất **~2–3,5 tiếng** trên T4 và xuất riêng `report_pack_{EXP}.zip`):
- `EXP = "f0"`: Baseline Faster R-CNN mặc định (`Default Anchors + CE + Cosine`)
- `EXP = "f1"`: `+ Small Sign Anchors (8..128, AR 0.75-1.30)`
- `EXP = "f2"`: `+ Warmup + ReduceLROnPlateau`
- `EXP = "f3"`: `+ CB-Focal + GIoU + RoIAlign 14x14` + tự chạy luôn **`F4` (Batch-4 SAHI) + FPN/Grad-CAM + Video Demo**
- `EXP = "f5"`: `Tiled 640x640 + Copy-Paste lớp hiếm + Batch-4 SAHI`

In [ ]:
EXP = "f0"  # Chọn 1 trong: "f0", "f1", "f2", "f3", "f5"
BRANCH = "minh"
REPO_URL = "https://github.com/dohoangminh-vn/AIT3001.Group6.traffic.sign.detection.git"

%cd /kaggle/working
!rm -rf btl && git clone -b {BRANCH} {REPO_URL} btl
%cd /kaggle/working/btl
!pip install -q pycocotools sahi albumentations pyyaml
!python -c "import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"

In [ ]:
import os
from pathlib import Path

matches = list(Path("/kaggle/input").rglob("train_traffic_sign_dataset.json"))
assert matches, "Không tìm thấy train_traffic_sign_dataset.json trong /kaggle/input! Hãy Add Input dataset trước."
RAW_DIR = str(matches[0].parent)
IMG_DIR = f"{RAW_DIR}/images"
print(f"Detected RAW_DIR = {RAW_DIR} ({len(list(Path(IMG_DIR).glob('*.png')))} images)")

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B API key loaded -> logging ONLINE.")
except Exception as e:
    print("W&B secret chưa gắn -> fallback OFFLINE + TensorBoard:", e)

In [ ]:
# Chuẩn hóa dữ liệu dùng chung (SHA256=e998588c9ab7 cố định từ configs/split_street_id.json)
if EXP == "f0":
    !python -u scripts/01_eda.py data.raw_dir={RAW_DIR}
!python -u scripts/02_prepare_data.py data.raw_dir={RAW_DIR}
!python -m pytest -q

In [ ]:
# Huấn luyện đúng 1 mô hình EXP + Đánh giá + Nén report_pack_{EXP}.zip
if EXP == "f0":
    !python -u scripts/04_train_frcnn.py img_dir={IMG_DIR} anchor_mode=default roi_size=7 loss_type=ce scheduler=cosine batch_size=4 grad_accum=1 out_dir=runs/frcnn/f0_default
    !python -u scripts/05_evaluate.py --img-dir {IMG_DIR} --check-oracle --preds F0_Default=outputs/preds/f0_default_val.json
    !zip -rq /kaggle/working/report_pack_f0.zip outputs/eda outputs/eval outputs/preds/f0_default_val.json runs/frcnn/f0_default/metrics_summary.json runs/frcnn/f0_default/resolved_config.yaml runs/frcnn/f0_default/tb_logs
elif EXP == "f1":
    !python -u scripts/04_train_frcnn.py img_dir={IMG_DIR} anchor_mode=small_sign_ar roi_size=7 loss_type=ce scheduler=cosine batch_size=4 grad_accum=1 out_dir=runs/frcnn/f1_small_ar
    !python -u scripts/05_evaluate.py --img-dir {IMG_DIR} --check-oracle --preds F1_SmallAR=outputs/preds/f1_small_ar_val.json
    !zip -rq /kaggle/working/report_pack_f1.zip outputs/eval outputs/preds/f1_small_ar_val.json runs/frcnn/f1_small_ar/metrics_summary.json runs/frcnn/f1_small_ar/resolved_config.yaml runs/frcnn/f1_small_ar/tb_logs
elif EXP == "f2":
    !python -u scripts/04_train_frcnn.py img_dir={IMG_DIR} anchor_mode=small_sign_ar roi_size=7 loss_type=ce scheduler=plateau batch_size=4 grad_accum=1 out_dir=runs/frcnn/f2_plateau
    !python -u scripts/05_evaluate.py --img-dir {IMG_DIR} --check-oracle --preds F2_Plateau=outputs/preds/f2_plateau_val.json
    !zip -rq /kaggle/working/report_pack_f2.zip outputs/eval outputs/preds/f2_plateau_val.json runs/frcnn/f2_plateau/metrics_summary.json runs/frcnn/f2_plateau/resolved_config.yaml runs/frcnn/f2_plateau/tb_logs
elif EXP == "f3":
    !python -u scripts/04_train_frcnn.py img_dir={IMG_DIR} out_dir=runs/frcnn/f3_focal_roi14
    !python -u scripts/06_infer_sahi_demo.py img_dir={IMG_DIR} weights=runs/frcnn/f3_focal_roi14/best.pth
    !python -u scripts/05_evaluate.py --img-dir {IMG_DIR} --check-oracle --preds F3_CBFocal_RoI14=outputs/preds/f3_focal_roi14_val.json F4_SAHI=outputs/preds/f3_focal_roi14_sahi_val.json
    !python -u scripts/07_analyze_and_demo.py --mode features img_dir={IMG_DIR} weights=runs/frcnn/f3_focal_roi14/best.pth
    !python -u scripts/07_analyze_and_demo.py --mode video img_dir={IMG_DIR} weights=runs/frcnn/f3_focal_roi14/best.pth
    !zip -rq /kaggle/working/report_pack_f3.zip outputs/eval outputs/preds/f3_focal_roi14_val.json outputs/preds/f3_focal_roi14_sahi_val.json runs/frcnn/f3_focal_roi14/metrics_summary.json runs/frcnn/f3_focal_roi14/resolved_config.yaml runs/frcnn/f3_focal_roi14/tb_logs
elif EXP == "f5":
    !python -u scripts/04_train_frcnn.py img_dir={IMG_DIR} tiled=true copy_paste=true min_size=800 max_size=2048 out_dir=runs/frcnn/f5_tiled640
    !python -u scripts/06_infer_sahi_demo.py img_dir={IMG_DIR} weights=runs/frcnn/f5_tiled640/best.pth
    !python -u scripts/05_evaluate.py --img-dir {IMG_DIR} --check-oracle --preds F5_Tiled=outputs/preds/f5_tiled640_val.json F5_Tiled_SAHI=outputs/preds/f5_tiled640_sahi_val.json
    !zip -rq /kaggle/working/report_pack_f5.zip outputs/eval outputs/preds/f5_tiled640_val.json outputs/preds/f5_tiled640_sahi_val.json runs/frcnn/f5_tiled640/metrics_summary.json runs/frcnn/f5_tiled640/resolved_config.yaml runs/frcnn/f5_tiled640/tb_logs
else:
    raise ValueError(f"EXP phải là f0/f1/f2/f3/f5, nhận được: {EXP}")

!ls -lh /kaggle/working/report_pack_*.zip